In [ ]:
## Import necessary libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Load datasets
train = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
test = pd.read_csv('/kaggle/input/nlp-getting-started/test.csv')
# sub = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')

# # Merge test data and submission to get texts and targets for test set
# merged_df = pd.merge(test, sub, on='id')

# test_X = merged_df['text']
# test_y = merged_df['target']

# print(train.shape, test.shape, sub.shape)

print(train.shape, test.shape)

# Reset index of training data for consistency
train = train.reset_index()

# Separate features and target
X = train['text']
y = train['target']

# Split train data into training and validation sets with stratification to preserve class distribution
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('stopwords')

stemmer = PorterStemmer()

def work(data):
    corpus = []

    for i in range(len(data)):
        review = re.sub('[^A-Za-z0-9]', ' ', data.iloc[i])
        review = review.lower()
        review = review.split()
        review = [stemmer.stem(word) for word in review if word not in set(stopwords.words('english'))]
        review = ' '.join(review)
        corpus.append(review)
    return corpus


X_train_processed = work(X_train)
X_train_array = np.array(X_train_processed)

X_test_processed = work(X_test)
X_test_array = np.array(X_test_processed)

# test_X_processed = work(test_x)
# test_X_array = np.array(test_X_processed)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

models = {
    'logistic':LogisticRegression(),
    'svm':SVC(),
    'randomforest':RandomForestClassifier()
}

"---"

from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()

X_train_cv = cv.fit_transform(X_train_array)
X_test_cv = cv.transform(X_test_array)
# test_X_cv = cv.transform(test_X_array)

### Model Building

from sklearn.metrics import accuracy_score, classification_report

for model_name, model in models.items():
    print(f"----- {model_name} -----")
    mod = model.fit(X_train_cv, y_train)
    
    # Train set predictions
    y_pred_train = model.predict(X_train_cv)
    train_acc = accuracy_score(y_train, y_pred_train)
    print("Train accuracy score:", train_acc)
    print("Train classification report:")
    print(classification_report(y_train, y_pred_train))
    
    # Test set predictions
    y_pred_test = model.predict(X_test_cv)
    test_acc = accuracy_score(y_test, y_pred_test)
    print("Test accuracy score:", test_acc)
    print("Test classification report:")
    print(classification_report(y_test, y_pred_test))
    print()  # For better spacing between model outputs

# from sklearn.metrics import accuracy_score, classification_report

# for model_name, model in models.items():
    
#     print(f"----- {model_name} -----")
    
    # y_pred_test = model.predict(test_X_cv)
    # test_acc = accuracy_score(test_y, y_pred_test)
    # print("Test accuracy score:", test_acc)
    # print("Test classification report:")
    # print(classification_report(test_y, y_pred_test))

In [ ]:
regressor = LogisticRegression()
regressor.fit(X_train_cv, y_train)

# Train set predictions
y_pred_train = regressor.predict(X_train_cv)
train_acc = accuracy_score(y_train, y_pred_train)
print("Train accuracy score:", train_acc)
print("Train classification report:")
print(classification_report(y_train, y_pred_train))
    
# Test set predictions
y_pred_test = regressor.predict(X_test_cv)
test_acc = accuracy_score(y_test, y_pred_test)
print("Test accuracy score:", test_acc)
print("Test classification report:")
print(classification_report(y_test, y_pred_test))
print()  # For better spacing between model outputs

In [ ]:
test = test['text']

In [ ]:
# Preprocessing Text Data > Text Data Tokenization
# Text Vectorization

# Preprocessing text data
text_preprocessed = work(test)
# List to array convertion
test_array = np.array(text_preprocessed)
# Text Vectorization
test_cv = cv.transform(test_array)
# Prediction 
test_pred = regressor.predict(test_cv)

print(test_pred)

In [ ]:
import pandas as pd

# Convert the NumPy array to a DataFrame
pd.DataFrame(test_pred, columns=['target']).to_csv('submission.csv', index=False)
